# Module 7: Advanced Policy Optimization — PPO from Scratch

## Learning Objectives
By completing this notebook, you will:
1. Explain why vanilla policy gradient methods are sensitive to step size and can suffer catastrophic policy collapse
2. Implement PPO-Clip with Generalized Advantage Estimation (GAE) in PyTorch
3. Visualize the clipping mechanism by inspecting probability ratio histograms
4. Tune the clip epsilon and observe its effect on update magnitude and training stability

## Prerequisites
- `01_reinforce_from_scratch.ipynb` and `02_actor_critic.ipynb` (Module 6)
- Understanding of policy gradient theorem and TD advantage

## Estimated Time: 15 minutes

---

## The Step-Size Problem

Policy gradient methods are notoriously sensitive to learning rate. A step that is too large can collapse the policy — the new policy behaves so differently from the data-collecting policy that all future estimates are invalid. Trust Region Policy Optimization (TRPO, Schulman 2015) solved this with a hard KL constraint, but it requires expensive second-order optimization.

**PPO-Clip** (Schulman et al., 2017) achieves a similar effect with a simple first-order trick. Define the probability ratio:

$$r_t(\theta) = \frac{\pi_\theta(a_t | s_t)}{\pi_{\theta_{\text{old}}}(a_t | s_t)}$$

The clipped surrogate objective is:

$$\mathcal{L}^{\text{CLIP}}(\theta) = \mathbb{E}_t \left[ \min\left( r_t(\theta) \hat{A}_t,\ \text{clip}(r_t(\theta), 1-\epsilon, 1+\epsilon) \hat{A}_t \right) \right]$$

The clip prevents $r_t$ from moving far from 1.0, which bounds the policy update magnitude. Typical $\epsilon = 0.2$.

In [ ]:
learning_objectives([
    "`01_reinforce_from_scratch.ipynb` and `02_actor_critic.ipynb` (Module 6)",
    "Understanding of policy gradient theorem and TD advantage",
    "## The Step-Size Problem",
    "(Schulman et al., 2017) achieves a similar effect with a simple first-order trick. Define the probability ratio:"
])

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

In [ ]:
apply_course_theme()
apply_plot_theme()

In [ ]:
# Purpose: Import dependencies and set reproducibility
# Key Concept: PPO uses rollout buffers, so we need slightly more infrastructure than actor-critic

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Categorical
import gymnasium as gym
import matplotlib.pyplot as plt
from collections import deque
import warnings

warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

env_probe = gym.make('CartPole-v1')
OBS_DIM = env_probe.observation_space.shape[0]  # 4
ACT_DIM = env_probe.action_space.n              # 2
env_probe.close()

print(f"CartPole-v1 | obs_dim={OBS_DIM} | act_dim={ACT_DIM}")

## 1. Actor-Critic Network for PPO

PPO uses the same actor-critic architecture we built in Module 6, but with a shared trunk for efficiency. The key difference from vanilla actor-critic is that we **freeze the old policy** after collecting a rollout and use it to compute importance ratios.

In [ ]:
# Purpose: Define shared-trunk actor-critic for PPO
# Key Concept: We need to compute old log-probs during rollout collection and new log-probs during update

class PPOActorCritic(nn.Module):
    """
    Shared-trunk actor-critic network for PPO.

    Parameters
    ----------
    obs_dim : int
        Observation space dimension.
    act_dim : int
        Number of discrete actions.
    hidden_dim : int
        Width of the shared trunk.
    """

    def __init__(self, obs_dim: int, act_dim: int, hidden_dim: int = 128):
        super().__init__()
        self.trunk = nn.Sequential(
            nn.Linear(obs_dim, hidden_dim),
            nn.Tanh()   # Tanh is standard in PPO implementations (bounded activations)
        )
        self.actor_head = nn.Linear(hidden_dim, act_dim)
        self.critic_head = nn.Linear(hidden_dim, 1)

    def forward(self, obs: torch.Tensor) -> tuple:
        """Return (Categorical distribution, value tensor)."""
        features = self.trunk(obs)
        dist = Categorical(logits=self.actor_head(features))
        value = self.critic_head(features).squeeze(-1)
        return dist, value

    def act(self, obs: torch.Tensor) -> tuple:
        """
        Sample action and return (action, log_prob, value).
        Used during rollout collection.
        """
        with torch.no_grad():
            dist, value = self.forward(obs)
        action = dist.sample()
        return action, dist.log_prob(action), value


# Sanity check
net = PPOActorCritic(OBS_DIM, ACT_DIM)
dummy = torch.zeros(1, OBS_DIM)
a, lp, v = net.act(dummy)
print(f"Action: {a.item()} | Log-prob: {lp.item():.4f} | Value: {v.item():.4f}")

## 2. Generalized Advantage Estimation (GAE)

One-step TD advantage has high bias; Monte Carlo advantage has high variance. GAE interpolates between them with parameter $\lambda$:

$$\hat{A}_t^{\text{GAE}(\gamma, \lambda)} = \sum_{l=0}^{\infty} (\gamma \lambda)^l \delta_{t+l}$$

where $\delta_t = r_t + \gamma V(s_{t+1}) - V(s_t)$ is the one-step TD error.

- $\lambda = 0$: pure one-step TD (low variance, high bias)
- $\lambda = 1$: Monte Carlo (high variance, no bias)
- $\lambda = 0.95$: the PPO default

This is computed backwards in $O(T)$ time, identical to computing discounted returns.

In [ ]:
# Purpose: Implement Generalized Advantage Estimation
# Key Concept: Backward scan accumulates advantages just like discounted returns

def compute_gae(
    rewards: list,
    values: list,
    dones: list,
    next_value: float,
    gamma: float = 0.99,
    lam: float = 0.95
) -> tuple:
    """
    Compute GAE advantages and returns for a rollout buffer.

    Parameters
    ----------
    rewards : list of float
        Rewards at each step.
    values : list of float
        Critic value estimates at each step.
    dones : list of bool
        Episode termination flags.
    next_value : float
        Bootstrap value from state after the last step.
    gamma : float
        Discount factor.
    lam : float
        GAE lambda parameter.

    Returns
    -------
    tuple of (advantages, returns) as torch.Tensors
    """
    advantages = []
    gae = 0.0

    # Extend values list with bootstrap value
    values_ext = values + [next_value]

    # Backward scan
    for t in reversed(range(len(rewards))):
        mask = 1.0 - float(dones[t])
        delta = rewards[t] + gamma * values_ext[t + 1] * mask - values_ext[t]
        gae = delta + gamma * lam * mask * gae
        advantages.insert(0, gae)

    advantages = torch.tensor(advantages, dtype=torch.float32)
    returns = advantages + torch.tensor(values, dtype=torch.float32)

    # Normalize advantages
    advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

    return advantages, returns


# Verify: short episode, all rewards = 1, no termination
test_adv, test_ret = compute_gae(
    rewards=[1.0, 1.0, 1.0],
    values=[0.5, 0.5, 0.5],
    dones=[False, False, True],
    next_value=0.0
)
print(f"GAE advantages: {test_adv.numpy().round(4)}")
print(f"Returns:        {test_ret.numpy().round(4)}")

## 3. Rollout Buffer

PPO collects a batch of $N$ steps (often across multiple environments), then performs $K$ gradient update epochs on the same batch. We store all transitions in a buffer before updating.

In [ ]:
# Purpose: Implement rollout buffer for collecting PPO training data
# Key Concept: Storing old log-probs is critical for computing the importance ratio r_t(theta)

class RolloutBuffer:
    """
    Fixed-size buffer for storing PPO rollout transitions.

    Stores observations, actions, old log-probs, rewards, values, and dones.
    Computes GAE advantages on demand.
    """

    def __init__(self):
        self.obs = []
        self.actions = []
        self.log_probs = []
        self.rewards = []
        self.values = []
        self.dones = []

    def store(self, obs, action, log_prob, reward, value, done):
        self.obs.append(obs)
        self.actions.append(action)
        self.log_probs.append(log_prob)
        self.rewards.append(reward)
        self.values.append(value)
        self.dones.append(done)

    def get(
        self,
        next_value: float,
        gamma: float = 0.99,
        lam: float = 0.95
    ) -> dict:
        """
        Return tensors for PPO update.

        Returns
        -------
        dict with keys: obs, actions, old_log_probs, advantages, returns
        """
        advantages, returns = compute_gae(
            self.rewards, self.values, self.dones, next_value, gamma, lam
        )
        return {
            'obs': torch.tensor(np.array(self.obs), dtype=torch.float32),
            'actions': torch.tensor(self.actions, dtype=torch.long),
            'old_log_probs': torch.tensor(self.log_probs, dtype=torch.float32),
            'advantages': advantages,
            'returns': returns
        }

    def clear(self):
        self.__init__()


print("RolloutBuffer defined.")

## 4. PPO Clipped Update

The clipped surrogate loss is the heart of PPO. We also add:
- **Critic loss** (MSE between predicted and target returns)
- **Entropy bonus** (encourages exploration)

The combined loss is:
$$\mathcal{L} = -\mathcal{L}^{\text{CLIP}} + c_1 \mathcal{L}^{\text{VF}} - c_2 H[\pi_\theta]$$

In [ ]:
# Purpose: Implement the PPO-Clip update step
# Key Concept: The min() in the clipped objective is the key — it pessimistically clips

def ppo_update(
    batch: dict,
    network: PPOActorCritic,
    optimizer: optim.Optimizer,
    epsilon: float = 0.2,
    vf_coef: float = 0.5,
    ent_coef: float = 0.01,
    n_epochs: int = 4
) -> dict:
    """
    Perform K epochs of PPO-Clip updates on a rollout batch.

    Parameters
    ----------
    batch : dict
        From RolloutBuffer.get().
    network : PPOActorCritic
        Shared actor-critic network.
    optimizer : optim.Optimizer
        Optimizer for network parameters.
    epsilon : float
        Clip range for probability ratio.
    vf_coef : float
        Value function loss coefficient.
    ent_coef : float
        Entropy bonus coefficient.
    n_epochs : int
        Number of gradient update epochs per batch.

    Returns
    -------
    dict with keys: 'policy_loss', 'value_loss', 'entropy', 'clip_fraction'
    """
    obs = batch['obs']
    actions = batch['actions']
    old_log_probs = batch['old_log_probs']
    advantages = batch['advantages']
    returns = batch['returns']

    metrics = {'policy_loss': [], 'value_loss': [], 'entropy': [], 'clip_fraction': []}

    for _ in range(n_epochs):
        # Recompute log-probs and values with current network parameters
        dist, values = network(obs)
        new_log_probs = dist.log_prob(actions)
        entropy = dist.entropy().mean()

        # Probability ratio: new_pi / old_pi in log space → exp(new - old)
        ratio = torch.exp(new_log_probs - old_log_probs)

        # Clipped surrogate objective
        surr1 = ratio * advantages
        surr2 = torch.clamp(ratio, 1.0 - epsilon, 1.0 + epsilon) * advantages
        policy_loss = -torch.min(surr1, surr2).mean()

        # Value function loss
        value_loss = nn.functional.mse_loss(values, returns)

        # Combined loss
        total_loss = policy_loss + vf_coef * value_loss - ent_coef * entropy

        optimizer.zero_grad()
        total_loss.backward()
        # Gradient clipping for additional stability
        nn.utils.clip_grad_norm_(network.parameters(), max_norm=0.5)
        optimizer.step()

        # Track what fraction of ratios were clipped (diagnostic)
        clipped = ((ratio < 1 - epsilon) | (ratio > 1 + epsilon)).float().mean()

        metrics['policy_loss'].append(policy_loss.item())
        metrics['value_loss'].append(value_loss.item())
        metrics['entropy'].append(entropy.item())
        metrics['clip_fraction'].append(clipped.item())

    return {k: np.mean(v) for k, v in metrics.items()}


print("PPO update function defined.")

## 5. Full PPO Training Loop

PPO collects `rollout_steps` transitions, computes GAE, then runs K epochs of gradient updates. We track clip fraction to see how often the optimization is constrained.

In [ ]:
# Purpose: Full PPO training loop on CartPole-v1
# Key Concept: Rollout steps separate data collection from gradient updates

def train_ppo(
    total_steps: int = 80_000,
    rollout_steps: int = 512,
    lr: float = 3e-4,
    gamma: float = 0.99,
    lam: float = 0.95,
    epsilon: float = 0.2,
    n_epochs: int = 4,
    seed: int = SEED
) -> dict:
    """
    Train PPO-Clip on CartPole-v1.

    Returns
    -------
    dict with keys: 'episode_rewards', 'clip_fractions', 'ratios_final_batch'
    """
    torch.manual_seed(seed)
    np.random.seed(seed)

    env = gym.make('CartPole-v1')
    network = PPOActorCritic(OBS_DIM, ACT_DIM)
    optimizer = optim.Adam(network.parameters(), lr=lr, eps=1e-5)
    buffer = RolloutBuffer()

    episode_rewards = []
    clip_fractions = []
    current_ep_reward = 0.0
    last_ratios = None

    obs, _ = env.reset()

    for step in range(total_steps):
        # Collect one transition
        obs_t = torch.tensor(obs, dtype=torch.float32).unsqueeze(0)
        action, log_prob, value = network.act(obs_t)

        next_obs, reward, terminated, truncated, _ = env.step(action.item())
        done = terminated or truncated
        current_ep_reward += reward

        buffer.store(
            obs, action.item(), log_prob.item(),
            reward, value.item(), done
        )

        obs = next_obs

        if done:
            episode_rewards.append(current_ep_reward)
            current_ep_reward = 0.0
            obs, _ = env.reset()

        # Update after collecting rollout_steps transitions
        if (step + 1) % rollout_steps == 0:
            # Bootstrap value for the last state
            with torch.no_grad():
                next_obs_t = torch.tensor(obs, dtype=torch.float32).unsqueeze(0)
                _, next_value = network(next_obs_t)
                next_value = next_value.item() * (1.0 - float(done))

            batch = buffer.get(next_value, gamma, lam)

            # Save ratios from the final batch for visualization
            with torch.no_grad():
                dist, _ = network(batch['obs'])
                new_lp = dist.log_prob(batch['actions'])
                last_ratios = torch.exp(new_lp - batch['old_log_probs']).numpy()

            metrics = ppo_update(batch, network, optimizer, epsilon, n_epochs=n_epochs)
            clip_fractions.append(metrics['clip_fraction'])

            buffer.clear()

    env.close()
    return {
        'episode_rewards': episode_rewards,
        'clip_fractions': clip_fractions,
        'ratios_final_batch': last_ratios
    }


print("Training PPO (epsilon=0.2)...")
ppo_results_02 = train_ppo(total_steps=80_000, epsilon=0.2)
print(f"Episodes completed: {len(ppo_results_02['episode_rewards'])}")
print(f"Final 10-ep avg reward: {np.mean(ppo_results_02['episode_rewards'][-10:]):.1f}")

## 6. Comparing Different Epsilon Values

Smaller $\epsilon$ = more conservative updates. Larger $\epsilon$ = closer to vanilla policy gradient. We compare $\epsilon \in \{0.1, 0.2, 0.3\}$ to see the tradeoff.

In [ ]:
# Purpose: Train PPO with different epsilon values
# Key Concept: Epsilon is the key hyperparameter controlling the trust region size

print("Training PPO (epsilon=0.1)...")
ppo_results_01 = train_ppo(total_steps=80_000, epsilon=0.1, seed=SEED)

print("Training PPO (epsilon=0.3)...")
ppo_results_03 = train_ppo(total_steps=80_000, epsilon=0.3, seed=SEED)

for eps, res in [(0.1, ppo_results_01), (0.2, ppo_results_02), (0.3, ppo_results_03)]:
    final_avg = np.mean(res['episode_rewards'][-10:]) if len(res['episode_rewards']) >= 10 else 0
    print(f"epsilon={eps}: final 10-ep avg = {final_avg:.1f}")

## 7. Visualizing Clip Behavior

In [ ]:
# Purpose: Visualize ratio distributions and clip fractions over training
# Key Concept: If no ratios are clipped, epsilon is too large; if most are clipped, too small

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# --- Plot 1: Reward curves for all epsilon values ---
ax = axes[0]
window = 10

for eps, res, color in [
    (0.1, ppo_results_01, 'steelblue'),
    (0.2, ppo_results_02, 'darkorange'),
    (0.3, ppo_results_03, 'crimson')
]:
    rewards = res['episode_rewards']
    if len(rewards) >= window:
        smoothed = np.convolve(rewards, np.ones(window) / window, mode='valid')
        ax.plot(smoothed, label=f'epsilon={eps}', color=color, linewidth=2)

ax.axhline(y=475, color='green', linestyle='--', alpha=0.7, label='Near-optimal')
ax.set_xlabel('Episode', fontsize=12)
ax.set_ylabel('Total Reward (10-ep avg)', fontsize=12)
ax.set_title('PPO: Effect of Clip Epsilon', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# --- Plot 2: Clip fraction over training (epsilon=0.2) ---
ax = axes[1]
clip_smooth = np.convolve(
    ppo_results_02['clip_fractions'],
    np.ones(5) / 5, mode='valid'
)
ax.plot(clip_smooth, color='darkorange', linewidth=2)
ax.axhline(y=0.0, color='black', linewidth=0.5)
ax.set_xlabel('Update iteration', fontsize=12)
ax.set_ylabel('Clip fraction', fontsize=12)
ax.set_title('Fraction of Updates Clipped\n(epsilon=0.2)', fontsize=13)
ax.grid(True, alpha=0.3)

# --- Plot 3: Probability ratio histogram from final batch ---
ax = axes[2]
ratios = ppo_results_02['ratios_final_batch']
if ratios is not None:
    ax.hist(ratios, bins=40, color='darkorange', edgecolor='white', alpha=0.8)
    ax.axvline(x=1.0, color='black', linewidth=2, label='ratio=1 (no change)')
    ax.axvline(x=0.8, color='red', linewidth=2, linestyle='--', label='clip bounds (eps=0.2)')
    ax.axvline(x=1.2, color='red', linewidth=2, linestyle='--')
ax.set_xlabel('Probability ratio r_t(theta)', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_title('Ratio Distribution (final batch)', fontsize=13)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('ppo_clip_analysis.png', dpi=100, bbox_inches='tight')
plt.show()
print("Figure saved to ppo_clip_analysis.png")

## Exercise 7.1: GAE Lambda Sensitivity

**Task:** GAE lambda controls the bias-variance tradeoff of advantage estimates. Train PPO with `lam` in `{0.0, 0.5, 0.95, 1.0}` (keep epsilon=0.2, total_steps=60_000). Report final 10-episode average reward for each.

**Expected observation:** Very low lambda (pure TD) and very high lambda (Monte Carlo) should both underperform the default 0.95.

<details>
<summary>Hint 1</summary>
Call `train_ppo(total_steps=60_000, lam=l, seed=42)` in a loop.
</details>

<details>
<summary>Hint 2 (more specific)</summary>
lam=0.0 means pure one-step TD advantage: delta_t = r_t + gamma*V(s') - V(s). This is computed correctly by compute_gae with lam=0 since the gae term vanishes.
</details>

In [ ]:
# YOUR CODE HERE
# ---------------
lambda_values = [0.0, 0.5, 0.95, 1.0]
lambda_results = {}  # Map lam -> final_10ep_avg_reward

# Fill lambda_results
# Print a summary

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------
def test_exercise_7_1():
    assert len(lambda_results) == 4, \
        f"Expected 4 lambda experiments, got {len(lambda_results)}"
    for lam in lambda_values:
        assert lam in lambda_results, f"Missing result for lam={lam}"
        val = lambda_results[lam]
        assert isinstance(val, (int, float)), \
            f"lambda_results[{lam}] should be a scalar, got {type(val)}"
        assert val >= 0, f"Reward must be non-negative, got {val}"
    print("Exercise 7.1 passed! GAE lambda sensitivity analysis complete.")

test_exercise_7_1()

## Exercise 7.2: Clip Fraction Diagnostic

**Task:** Write a function `compute_clip_fraction(ratios, epsilon)` that takes a numpy array of probability ratios and an epsilon value, and returns the fraction of ratios that fall outside `[1-epsilon, 1+epsilon]`.

**Expected output:** For ratios all equal to 1.0, the clip fraction should be 0.0. For ratios all equal to 2.0 with epsilon=0.2, the fraction should be 1.0.

<details>
<summary>Hint 1</summary>
Use boolean indexing: `(ratios < 1 - epsilon) | (ratios > 1 + epsilon)` gives a boolean array.
</details>

In [ ]:
# YOUR CODE HERE
# ---------------

def compute_clip_fraction(ratios: np.ndarray, epsilon: float) -> float:
    """
    Compute the fraction of probability ratios that fall outside [1-epsilon, 1+epsilon].

    Parameters
    ----------
    ratios : np.ndarray
        Probability ratios r_t(theta) = new_pi / old_pi.
    epsilon : float
        Clip range.

    Returns
    -------
    float
        Fraction in [0, 1].
    """
    your_answer = None  # Replace with your implementation
    return your_answer

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------
def test_exercise_7_2():
    # All ratios at 1.0 — nothing clipped
    r_no_clip = np.ones(100)
    result = compute_clip_fraction(r_no_clip, 0.2)
    assert result is not None, "Function must return a value, not None"
    assert abs(result - 0.0) < 1e-6, f"Expected 0.0, got {result}"

    # All ratios at 2.0 — all clipped (2.0 > 1.2)
    r_all_clip = np.full(100, 2.0)
    result = compute_clip_fraction(r_all_clip, 0.2)
    assert abs(result - 1.0) < 1e-6, f"Expected 1.0, got {result}"

    # Half inside, half outside
    r_half = np.concatenate([np.ones(50), np.full(50, 2.0)])
    result = compute_clip_fraction(r_half, 0.2)
    assert abs(result - 0.5) < 1e-6, f"Expected 0.5, got {result}"

    print("Exercise 7.2 passed! Clip fraction computation correct.")

test_exercise_7_2()

In [ ]:
section_divider("Key Takeaways")

## Key Takeaways

1. **PPO-Clip prevents destructive policy updates** by constraining the probability ratio to `[1-epsilon, 1+epsilon]`. This approximates the TRPO trust region without expensive second-order optimization.

2. **GAE bridges TD and Monte Carlo advantages** using lambda. The default `lambda=0.95` gives low-variance estimates while maintaining reasonable bias.

3. **Clip fraction is a diagnostic**: too low means epsilon is too large (updates are unconstrained); too high means epsilon is too small (updates are over-constrained). Typical healthy clip fraction is 5-20%.

4. **Multiple gradient epochs per rollout** (the inner loop) is what makes PPO sample-efficient relative to pure on-policy methods. The clipping prevents overfitting the policy to the stale rollout data.

5. **Rollout step count matters**: too few steps means the batch is noisy; too many means the environment has changed too much by the time you update.

## What's Next

`02_algorithm_showdown.ipynb` puts REINFORCE, A2C, and PPO head-to-head on CartPole with statistical rigor: multiple seeds, identical hyperparameter budgets, and sample efficiency curves.

## Additional Resources

- Schulman et al. (2017): "Proximal Policy Optimization Algorithms" — the original PPO paper
- Schulman et al. (2015): "High-Dimensional Continuous Control Using Generalized Advantage Estimation"
- CleanRL: [PPO implementation](https://github.com/vwxyzjn/cleanrl) — a clean reference implementation

In [ ]:
key_takeaways([
    "Review the key concepts covered in this notebook",
    "Practice applying these techniques to your own data",
    "Connect this material to the companion guide and slides"
])